<a href="https://colab.research.google.com/github/omnhinge/TA0075/blob/main/deepfake_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchaudio transformers librosa numpy

The real Running code:

In [ ]:
import torch
import librosa
import numpy as np
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# 1. Load the Pre-trained Model and Feature Extractor
model_name = "Gustking/wav2vec2-large-xlsr-deepfake-audio-classification"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name)

def detect_voice_fake(audio_path):
    # Load and resample
    audio, sr = librosa.load(audio_path, sr=16000)

    # NEW: Normalize the audio to a standard volume range
    audio = librosa.util.normalize(audio)

    # NEW: Only take the first 5 seconds to ensure model focus.
    # audio = audio[:16000*5]

    # Extract Features
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt", padding=True)

    # Run Inference
    with torch.no_grad():
        logits = model(**inputs).logits

    # Convert Logits to Probability
    probs = torch.nn.functional.softmax(logits, dim=-1)

    # PRINT THIS to check which index is which
    # print(f"Raw Probabilities: {probs}")

    # Let's return the whole tensor so we can inspect it
    return probs[0].tolist()

# Example Usage
audio_file = "/content/fake_audio.mp4"
probs = detect_voice_fake(audio_file)

fake_score = probs[0] * 100
real_score = probs[1] * 100
#here, Label 0 is 'Fake/Spoof' and Label 1 is 'Real'

print(f"--- Results for {audio_file} ---")
print(f"Fake Probability: {fake_score:.2f}%")
print(f"Real Probability: {real_score:.2f}%")

if fake_score > 49:
    print("⚠️ ALERT: Deepfake Detected")
else:
    print("✅ Audio appears authentic")